In [ ]:
import os
from dotenv import load_dotenv
import gradio as gr
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch
from pythainlp.tokenize import word_tokenize
from openai import OpenAI
import re
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# โหลด tokenizer และ model
model_name = "Porameht/wangchanberta-thainer-corpus-v2-2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)
label_list = model.config.id2label

def ner_highlight(text):
    # inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    cut = word_tokenize(text.replace(" ", "<_>"))
    inputs = tokenizer(cut, is_split_into_words=True, return_tensors="pt")
    ids = inputs["input_ids"]
    mask = inputs["attention_mask"]
    outputs = model(ids, attention_mask=mask)
    logits = outputs[0]
    predictions = torch.argmax(logits, dim=2)
    predicted_token_class = [model.config.id2label[t.item()] for t in predictions[0]]
    
    def fix_span_error(words, ner):
        _ner_tag = []
        for i, j in zip(words, ner):
            i = tokenizer.decode(i)
            if i.isspace() and j.startswith("B-"):
                j = "O"
            if i == '' or i == '<s>' or i == '</s>':
                continue
            if i == "<_>":
                i = " "
            _ner_tag.append((i, j))
        return _ner_tag
    ner_tag = fix_span_error(inputs['input_ids'][0], predicted_token_class)

    html = ""
    for token, label in ner_tag:
        if token in tokenizer.all_special_tokens:
            continue
        word = token.replace("▁", " ")
        if label != "O":
            label_type = label.split("-")[-1]
            html += f"<mark style='background-color: #d1e7dd;' title='{label_type}'>{word}</mark>"
        else:
            html += word
    return html.strip()

def ner_mask(text):
    cut = word_tokenize(text.replace(" ", "<_>"))
    inputs = tokenizer(cut, is_split_into_words=True, return_tensors="pt")
    ids = inputs["input_ids"]
    mask = inputs["attention_mask"]
    outputs = model(ids, attention_mask=mask)
    logits = outputs[0]
    predictions = torch.argmax(logits, dim=2)
    predicted_token_class = [model.config.id2label[t.item()] for t in predictions[0]]
    
    def fix_span_error(words, ner):
        _ner_tag = []
        for i, j in zip(words, ner):
            i = tokenizer.decode(i)
            if i.isspace() and j.startswith("B-"):
                j = "O"
            if i == '' or i == '<s>' or i == '</s>':
                continue
            if i == "<_>":
                i = " "
            _ner_tag.append((i, j))
        return _ner_tag
    
    ner_tag = fix_span_error(inputs['input_ids'][0], predicted_token_class)
    
    masked = ""
    current_entity = None 
    
    for token, tag in ner_tag:
        if tag == "O":
            masked += token 
            current_entity = None
        else:
            entity = tag.split('-')[1]
            if tag.startswith('B-') or current_entity != entity:
                masked += f"[{entity}]"
                current_entity = entity
            elif tag.startswith('I-') and current_entity == entity:
                continue
            else:
                current_entity = None
                masked += token
    return re.sub(r'(\[[A-Z]+\])\1+', r'\1', masked.strip())


def summarize_issue(masked_text):
    system_prompt = """คุณคือผู้ช่วยวิเคราะห์ VOC ที่เน้นหาประเด็นปัญหาเชิงเนื้อหาแบบกระชับและเข้าใจง่ายในโทนภาษาที่เป็นกันเองเหมือนพูดกับผู้บริหาร"""
    user_prompt = f"""จากข้อความที่ผ่านการปกปิดข้อมูลส่วนบุคคลดังนี้:\n\n{masked_text}\n\nช่วยสรุปประเด็นปัญหาหลักที่เกิดขึ้นให้เข้าใจง่าย"""

    response = client.chat.completions.create(
        model="gpt-4o-mini-2024-07-18",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.1,
        max_tokens=1024,
    )
    return response.choices[0].message.content.strip()

def summarize_topic(masked_text):
    system_prompt = """คุณคือผู้ช่วยวิเคราะห์ VOC ที่สามารถจำแนกประเภทเสียงของลูกค้าได้ตามหมวดหมู่ต่อไปนี้:
1. แจ้งปัญหาด้านบริการ
2. แจ้งปัญหาคุณภาพไฟฟ้า
3. ชื่นชม
4. ร้องขอ
5. แจ้งเบาะแส

หากข้อความเข้าข่ายมากกว่าหนึ่งประเภทแบบมีความน่าจะเป็นสูงมากๆ ให้ระบุไม่เกิน 2 ประเภท และให้ตอบกลับโดย:
- ระบุประเภทเสียงของลูกค้าเป็นชื่อหมวดหมู่ (ตามรายการด้านบน)
- ตามด้วยเหตุผลสั้นๆ แบบเข้าใจง่ายในรูปแบบ: “เพราะ...”

ตอบให้กระชับ ชัดเจน และเป็นกันเองเหมือนพูดกับผู้บริหาร"""

    user_prompt = f"""จากข้อความที่ผ่านการปกปิดข้อมูลส่วนบุคคลดังนี้:\n\n{masked_text}\n\nช่วยจำแนกประเภทของเสียงของลูกค้า พร้อมเหตุผลแบบกระชับ"""

    response = client.chat.completions.create(
        model="gpt-4o-mini-2024-07-18",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.2,
        max_tokens=512,
    )
    return response.choices[0].message.content.strip()

def sentiment_score(summary_text):
    user_prompt = f"""จากข้อความนี้:\n\n{summary_text}\n\nช่วยให้คะแนนความพึงพอใจในช่วง -5 (ไม่พอใจอย่างรุนแรง) ถึง +5 (พึงพอใจมาก) พร้อมสรุปสั้น ๆ ว่าทำไมถึงได้คะแนนนั้น"""

    response = client.chat.completions.create(
        model="gpt-4o-mini-2024-07-18",
        messages=[
            {"role": "system", "content": "คุณคือผู้ช่วยให้คะแนนความพึงพอใจจากข้อความร้องเรียน"},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.2,
        max_tokens=512
    )
    return response.choices[0].message.content.strip()

def process(text):
    highlighted = ner_highlight(text)
    masked = ner_mask(text)
    summary = summarize_issue(masked)
    voc_type = summarize_topic(masked)
    scorecard = sentiment_score(summary)
    return highlighted, masked, summary, voc_type, scorecard

with gr.Blocks(css="""
.full-height textarea {
    height: auto !important;
    min-height: 200px;
}
""") as app:
    gr.Markdown("""
    # 🔒 Thai PII Anonymizer with NER (Custom Token Classification)
    ใช้ PyThaiNLP สำหรับตัดคำภาษาไทย และโมเดล NER แบบ custom เพื่อปกปิดข้อมูลส่วนบุคคล และสรุปประเด็นปัญหาเชิงเนื้อหา พร้อมวิเคราะห์ความพึงพอใจ
    """)

    with gr.Row():
        with gr.Column():
            input_text = gr.Textbox(lines=5, label="Input Text", placeholder="ใส่ข้อความที่ต้องการประมวลผล...")
            submit_btn = gr.Button("Submit", variant="primary")
        with gr.Column():
            voice_type_output = gr.Textbox(label="ประเภทเสียงของลูกค้า", lines=2, elem_classes="full-height")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### NER Result (Highlighted)")
            highlight_output = gr.HTML()
        with gr.Column():
            mask_output = gr.Textbox(label="Anonymized Output (Masking)", lines=10, elem_classes="full-height")

    gr.Markdown("### 📌 สรุปประเด็นปัญหาเชิงเนื้อหา")
    summary_output = gr.Textbox(lines=6, elem_classes="full-height")

    gr.Markdown("### 🧭 คะแนนความพึงพอใจจากข้อความ")
    sentiment_output = gr.Textbox(lines=3, elem_classes="full-height")

    submit_btn.click(fn=process, inputs=input_text, outputs=[highlight_output, mask_output, summary_output, voice_type_output, sentiment_output])

app.launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [2]:
app.close()

Closing server running on port: 7860
